![Machine Learning Lab](banner.jpg)

# Laboratorio 5

## Objetivos

1. Entender la correlación entre variables y cómo eliminar redundancias para mejorar los modelos.
2. Aplicar PCA para reducir la dimensionalidad y analizar la varianza explicada.
3. Determinar el número óptimo de componentes para preservar la mayor información posible.
4. Entrenar Naïve Bayes con diferentes cantidades de componentes y evaluar su desempeño.
5. Analizar el impacto de PCA en la precisión y F1-score del modelo.



In [ ]:
!mkdir datasets
!curl -L -o datasets/breast-cancer-wisconsin-data.zip https://www.kaggle.com/api/v1/datasets/download/uciml/breast-cancer-wisconsin-data
!unzip datasets/breast-cancer-wisconsin-data.zip -d datasets/breast-cancer-wisconsin-data

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
data = pd.read_csv("datasets/breast-cancer-wisconsin-data/data.csv").drop(columns=['id'])

correlation_matrix = data.select_dtypes(['number']).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, cmap="coolwarm", cbar=True)
plt.title("Correlation Heatmap of Breast Cancer Dataset")
plt.show()

## Repaso de correlación

Antes de comenzar con la práctica de laboratorio, repasemos un concepto que ya hemos aplicado en laboratorios anteriores: la eliminación de variables con alta correlación.

La razón principal para realizar este proceso es evitar la presencia de variables colineales, ya que esto introduce información redundante en el modelo. Cuando varias variables están altamente correlacionadas, el modelo puede volverse menos interpretable y más propenso a sobreajuste.

Para comprender mejor este concepto, veamos el siguiente ejemplo.

In [ ]:
# Supongamos que tenemos dos variables dependientes X1 y X2 que son colineares
X1 = np.arange(10)
X2 = X1 / 2 + 10
plt.plot(X1, X2, 'o')
plt.show()

In [ ]:
# Ahora supongamos que tenemos un modelo y1
def y1(x1, x2):
    return x1 + x2


print(y1(X1, X2))


# Si ese es el caso ya que X2 es dependiente de X1 podemos hacer un nuevo modelo y2 al remplazar X2 en y1
# y1 = x1 + x2 -> y1 = x1 + x1 / 2 + 10 = (3/2) * x1 + 10
def y2(x1, x2):
    return x1 + x1 / 2 + 10


print(y2(X1, X2))

print(y1(X1, X2) == y2(X1, X2))

Ahora podemos observar que es posible construir un modelo de **y** sin necesidad de utilizar la variable **X2**. Esto se debe a que **X2** no aporta información adicional más allá de la proporcionada por **X1**.  

Algo similar ocurre cuando varias variables presentan una alta correlación entre sí: algunas no añaden información nueva, ya que su contenido está implícito en otras variables. Por esta razón, las eliminamos, reduciendo tanto el **ruido** en los datos como el **costo computacional** asociado a procesar variables que no contribuyen significativamente al modelo.

## PCA  

¿Qué sucede cuando la colinealidad no se limita a solo dos variables, sino que involucra tres, cuatro o más? Por ejemplo, en un caso donde **X3** pueda expresarse como una combinación lineal de **X1** y **X2** (`X3 = A*X1 + B*X2 + C`). ¿Cómo podemos identificar esta redundancia de información?  

La respuesta es mediante **PCA (Análisis de Componentes Principales)**. PCA es una técnica que proyecta los datos en un espacio de menor dimensión utilizando combinaciones lineales de las variables originales, con el objetivo de **preservar la mayor cantidad de varianza posible**.



In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = scaler.fit_transform(data.select_dtypes(['number']).dropna(axis=1))

# Apply PCA
pca = PCA(n_components=10)
pca_components = pca.fit_transform(scaled_data)

pca_components

Como pueden notar, estamos escalando las variables utilizando **Standard Scaler**. Esto se debe a que **PCA se basa en la varianza de las variables**, lo que lo hace sensible a diferencias en la escala de los datos. Para evitar este efecto, es fundamental **normalizar las variables** antes de aplicar PCA, asegurando que todas contribuyan equitativamente al análisis.  

## Visualización  

Además, **PCA facilita la identificación de agrupaciones (clusters)** en conjuntos de datos de alta dimensionalidad, permitiendo una mejor interpretación y visualización de la estructura de los datos.

In [ ]:
# Solo usamos los primeros 2 componentes
data_pca = pd.DataFrame(pca_components[:, :2], columns=['PCA1', 'PCA2'])
data_pca['diagnosis'] = data['diagnosis'].replace(['M', 'B'], ['benign', 'malignant'])

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue=data_pca['diagnosis'], data=data_pca, palette='Set1', alpha=0.7)
plt.title('PCA Projection of Breast Cancer Dataset')
plt.show()

En PCA los componentes estan ordenados por cuanta de la varianza explican. Y tambien nos permite saber cuanta varianza explica cada componente

In [ ]:

explained_variance = pca.explained_variance_ratio_
plt.figure(figsize=(10, 6))
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, color='b', label='Individual Explained Variance')
plt.step(range(1, len(explained_variance) + 1), np.cumsum(explained_variance), where='mid', color='r', label='Cumulative Explained Variance')
plt.ylabel('Explained Variance Ratio')
plt.xlabel('Principal Components')
plt.title('Explained Variance by PCA Components')
plt.legend(loc='best')
plt.grid(True)
plt.show()

Por ejemplo podemos obtener cuantos componentes son necesarios para mantener 80% de la varianza

In [ ]:
number_of_components = 0
accumulated_variance = 0
for component_variance in explained_variance:
    accumulated_variance += component_variance
    number_of_components += 1
    if accumulated_variance >= 0.8:
        break

print('number of components', number_of_components)

pca_components[:, :number_of_components]

## Naïve Bayes  

Naïve Bayes es un algoritmo de clasificación basado en el **Teorema de Bayes**, el cual asume que las características son **independientes entre sí** (aunque esta suposición rara vez se cumple en la práctica). A pesar de esta limitación, es **altamente eficiente y escalable**, lo que lo hace especialmente útil en tareas como la **clasificación de texto** (por ejemplo, el filtrado de spam). Su funcionamiento se basa en el cálculo de **probabilidades condicionales** para realizar predicciones.  

Por defecto, **Naïve Bayes no admite variables continuas**. Para abordar este problema, utilizaremos **Gaussian Naïve Bayes**, que asume que las características siguen una **distribución normal**. En nuestro caso, podemos entrenar el modelo utilizando un número arbitrario de componentes, siempre menor que el número original de variables (*features*).

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


X_10 = pca_components[:, :10]
y = data['diagnosis'].replace(['M', 'B'], [1, 0])

X_10_train, X_10_test, y_10_train, y_10_test = train_test_split(X_10, y, test_size=0.2, random_state=42)

gnb_10 = GaussianNB()
gnb_10.fit(X_10_train, y_10_train)
y_pred_10 = gnb_10.predict(X_10_test)

print("Metrics for Gaussian Naive Bayes with 10 PCA components (Test Data):")
print(classification_report(y_10_test, y_pred_10, target_names=['Benign', 'Malignant']))

X_2 = pca_components[:, :2]

# Dividir los datos en train/test
X_2_train, X_2_test, y_2_train, y_2_test = train_test_split(X_2, y, test_size=0.2, random_state=42)

gnb_2 = GaussianNB()
gnb_2.fit(X_2_train, y_2_train)
y_pred_2 = gnb_2.predict(X_2_test)

print("Metrics for Gaussian Naive Bayes with 2 PCA components (Test Data):")
print(classification_report(y_2_test, y_pred_2, target_names=['Benign', 'Malignant']))